In [16]:
# 🔧 Config import
import logging
import sys
import os
from PySide6.QtWidgets import QApplication
import pandas as pd

# Set up project root in sys.path
try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Fallback for Jupyter or interactive shell
    current_dir = os.getcwd()

project_root = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.append(project_root)

# Internal imports
from Config.LoggerConfig import colored_logger
from Data.Data_Fetcher import DataFetcher
from UI.MainWindow import MainWindow

logger = colored_logger()
current_file = os.path.basename(__file__) if '__file__' in globals() else "Notebook"
logger.info(f"Logger initialized ({current_file})")

print("\033[92mThis should be light green\033[0m")


[2025-05-25 15:09:13] INFO - 3031238702.py - Logger initialized (Notebook)


This should be light green


In [17]:
 #📊 Data parameters
symbol = "BTCUSDT"
start_date = "01-01-2025"
end_date = "24-04-2025"
interval = "1m"

In [18]:
# 📥 Load data
try:
    logger.info(f"Initializing DataFetcher for {symbol} from {start_date} to {end_date} at {interval} interval.")
    fetcher = DataFetcher(symbol, start_date, end_date, interval)
    #fetcher.get_binance_data()
    #fetcher.save_to_csv(directory="./Data")
    fetcher.load_from_csv(directory="./Data")
    print(fetcher.raw_data.head())
    logger.info(f"✅ Data successfully loaded. Shape: {fetcher.raw_data.shape}")
except Exception as e:
    logger.error(f"❌ Failed to load data: {e}")


[2025-05-25 15:09:58] INFO - 3561617063.py - Initializing DataFetcher for BTCUSDT from 01-01-2025 to 24-04-2025 at 1m interval.
[2025-05-25 15:09:58] INFO - Data_Fetcher.py - 📥 Data loaded from: ./Data/BTCUSDT_01-01-2025_24-04-2025_1m.csv
[2025-05-25 15:09:58] INFO - 3561617063.py - ✅ Data successfully loaded. Shape: (162781, 6)


             timestamp     open     high      low    close   volume
0  2024-12-31 23:00:00  93469.1  93476.8  93419.1  93419.1   70.019
1  2024-12-31 23:01:00  93419.2  93453.4  93362.9  93401.9  121.720
2  2024-12-31 23:02:00  93401.9  93459.0  93392.1  93458.9   71.811
3  2024-12-31 23:03:00  93459.0  93550.9  93458.1  93484.5  133.483
4  2024-12-31 23:04:00  93484.5  93516.5  93365.5  93394.1   87.904


In [ ]:
# 🖥️ Start application
try:
    logger.info("Starting QApplication...")
    app = QApplication(sys.argv)

    window = MainWindow()
    logger.info("MainWindow initialized.")

    df = fetcher.raw_data[-3000:].copy()

    if 'timestamp' not in df.columns:
        logger.error("❌ 'timestamp' column missing from DataFrame.")
        sys.exit(1)

    df['timestamp'] = pd.to_datetime(df['timestamp'])
    logger.info("✅ 'timestamp' column converted to datetime format.")

    window.load_and_plot(df)
    logger.info("Candlestick plot loaded into MainWindow.")

    window.show()
    logger.info("MainWindow shown. Entering application loop.")
    sys.exit(app.exec())

except Exception as e:
    logger.exception(f"❌ Application startup failed: {e}")
    sys.exit(1)

